In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

print("Loading master features CSV...")
df = pd.read_csv(Path.cwd().parent /'master_dataset_lexical_features.csv')

Loading master features CSV...


In [13]:
X = df.drop(['url', 'label'], axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Data split successful. Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

Data split successful. Training shape: (468027, 16), Testing shape: (117007, 16)


In [ ]:
print("Initializing XGBoost Classifier...")

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    colsample_bytree=0.6,   # ← each tree only sees 60% of features
    colsample_bylevel=0.6,  # ← each split only sees 60% of features
    subsample=0.8,          # ← each tree trains on 80% of rows
    reg_alpha=0.1,          # ← L1 regularization, forces feature diversity
    min_child_weight=5, 
)

print("Training the sequential gradient boosting pipeline...")
model.fit(X_train, y_train)
print("XGBoost training cycle complete!")

Initializing XGBoost Classifier...
Training the sequential gradient boosting pipeline...
XGBoost training cycle complete!


In [15]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Safe', 'Malicious'])

print("\n================ XGBOOST METRICS ================")
print(f"Target Threshold Clear State: {accuracy * 100:.2f}%")
print("-------------------------------------------------")
print("Confusion Matrix Layout:")
print(cm)
print("-------------------------------------------------")
print("Classification Matrix Specs:")
print(report)


================ XGBOOST METRICS ================
Target Threshold Clear State: 95.81%
-------------------------------------------------
Confusion Matrix Layout:
[[59421   579]
 [ 4325 52682]]
-------------------------------------------------
Classification Matrix Specs:
              precision    recall  f1-score   support

        Safe       0.93      0.99      0.96     60000
   Malicious       0.99      0.92      0.96     57007

    accuracy                           0.96    117007
   macro avg       0.96      0.96      0.96    117007
weighted avg       0.96      0.96      0.96    117007



In [16]:
xgb_model = Path.cwd().parent /'ml_model_xgb.joblib'

joblib.dump(model, xgb_model)

['/Users/AmeyaWalekar/Desktop/Summer 2026/Phisguard - CC Project/The Project /phishguard/ml_model_xgb.joblib']

In [17]:
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
print(feat_imp.sort_values(ascending=False))

slash_count           0.968696
url_length            0.013744
dot_count             0.009492
brand_in_subdomain    0.003527
suspicious_tld        0.001537
digit_ratio           0.001275
hyphen_count          0.000695
has_port              0.000333
url_entropy           0.000252
hostname_length       0.000175
subdomain_count       0.000165
is_ip                 0.000109
at_count              0.000000
query_count           0.000000
path_depth            0.000000
is_url_shortener      0.000000
dtype: float32
